In [1]:
from lokigi.site import SiteProblem
import geopandas
import pandas as pd
import pickle
import imageio.v2 as imageio
from pathlib import Path
import matplotlib.pyplot as plt
import io
from PIL import Image
import plotly.express as px

c:\geographic_or_ds_playground\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
problem = SiteProblem()

problem.add_demand(
    pd.read_csv("demand_MF_50_84.csv"), demand_col="MF50-84", location_id_col="LSOA 2021 Name"
)

problem.add_region_geometry_layer(
    geopandas.read_file("LSOA_Devon_2021_EW_BSC_V4.gpkg"), common_col="LSOA21NM"
)

problem.add_equity_data("devon_imd_2025_2021_LSOAs.csv",
                        equity_col="Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOA",
                        common_col="LSOA name (2021)", direction="higher_is_better", label="IMD Decile")

existing_cdcs = pd.read_csv("devon_cdcs.csv")
existing_cdcs_gdf =     geopandas.GeoDataFrame(
        existing_cdcs,  # Our pandas dataframe
        geometry=geopandas.points_from_xy(
            existing_cdcs[
                "Longitude"
            ],  # Our 'x' column (horizontal position of points)
            existing_cdcs["Latitude"],  # Our 'y' column (vertical position of points)
        ),
        crs="EPSG:4326",
    )

problem.add_sites(
        existing_cdcs_gdf,
        candidate_id_col="Facility_Name",
        required_sites_col="Existing"
    )

problem_car = problem.copy()

problem_car.add_travel_matrix(
    pd.read_csv("travel_matrix_car.csv").fillna(9999.0), unit="minutes", source_col="from_id"
)

# problem_car.add_additional_data(
#     pd.read_csv("travel_matrix_public_transport.csv").fillna(9999.0), common_col="from_id", label="Public Transport Travel Time (minutes)",
#     column_of_interest=""
# )

problem_pt = problem.copy()

problem_pt.add_travel_matrix(
    pd.read_csv("travel_matrix_public_transport.csv").fillna(9999.0), unit="minutes", source_col="from_id"
)

# problem_pt.add_additional_data(
#     pd.read_csv("travel_matrix_car.csv").fillna(9999.0), common_col="from_id", label="Car Travel Time (minutes)"
# )

C:\Users\Sammi\AppData\Local\Temp\ipykernel_14848\639360519.py:11: FutureWarning: The 'direction' argument to add_equity_data() is deprecated. Use disadvantaged_end='low' instead of direction='higher_is_better' (lowest values = most deprived, e.g. DLUHC IMD deciles), and disadvantaged_end='high' instead of direction='higher_is_worse' (highest values = most deprived, e.g. raw IMD scores).
  problem.add_equity_data("devon_imd_2025_2021_LSOAs.csv",


In [3]:
solutions_car = []

for i in range(4, 16):
    solutions_car.append({'n': i, 'solution': problem_car.solve(p=i, threshold_for_coverage=30, n_jobs=-1)})

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 364/364 [00:02<00:00, 177.41it/s]


### Compare with best results for 1, 2 and 3 sites

We have to set these up slightly differently because this version of lokigi can't handle p being < number of sites marked as 'required'. 

In [4]:
problem_car_simple = problem_car.copy()

problem_car_simple.add_sites(
        existing_cdcs_gdf[existing_cdcs_gdf["Existing"]=="Yes"],
        candidate_id_col="Facility_Name",
    )

problem_pt_simple = problem_pt.copy()

problem_pt_simple.add_sites(
        existing_cdcs_gdf[existing_cdcs_gdf["Existing"]=="Yes"],
        candidate_id_col="Facility_Name",
    )

In [5]:
solution_1_car = problem_car_simple.solve(p=1, threshold_for_coverage=30)
solution_2_car = problem_car_simple.solve(p=2, threshold_for_coverage=30)
solution_3_car = problem_car_simple.solve(p=3, threshold_for_coverage=30)

  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:00<00:00, 35.41it/s]


In [6]:
solutions_car.append({'n': 1, 'solution':solution_1_car})
solutions_car.append({'n': 2, 'solution':solution_2_car})
solutions_car.append({'n': 3, 'solution':solution_3_car})

# Compare best

In [7]:
result_df_comparison = []

for sol in solutions_car:
    result_df_comparison.append(sol['solution'].return_best_combination_details(top_n=1))

result_df_comparison = pd.concat(result_df_comparison)
result_df_comparison['n'] = result_df_comparison['site_indices'].apply(lambda x: len(x))
result_df_comparison

,index,solution_rank,site_names,site_indices,coverage_threshold,weighted_average,unweighted_average,90th_percentile,max,total_cost,...,gap_relative_weighted,avg_lower_third_bins,avg_middle_third_bins,avg_upper_third_bins,inter_tertile_ratio,gap_absolute_description,gap_relative_description,inter_tertile_description,problem_df,n
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3]",30,23.392076,21.421445,42.396666,61.533333,NaN,...,2.400495,18.8675,26.060000,19.263333,0.979451,Spread of 17.0 units between best and worst gr...,Significant Disparity (Worst group travels 140...,Balanced (Macro travel times are broadly equal),LSOA 2021 Name Bideford Community Hospi...,4
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 12]",30,21.933916,20.026795,39.406667,61.533333,NaN,...,2.757214,16.9675,24.676667,18.693333,0.907677,Spread of 17.7 units between best and worst gr...,Significant Disparity (Worst group travels 176...,Slightly Progressive (Most deprived travel 9% ...,LSOA 2021 Name Bideford Community Hospi...,5
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 5, 12]",30,20.641178,19.009099,34.430001,61.533333,NaN,...,2.540299,15.6625,23.536667,18.646667,0.839962,Spread of 15.5 units between best and worst gr...,Significant Disparity (Worst group travels 154...,Slightly Progressive (Most deprived travel 16%...,LSOA 2021 Name Bideford Community Hospi...,6
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 5, 8, 12]",30,19.415423,17.810882,33.533333,56.350000,NaN,...,2.373134,14.9125,22.146667,17.423333,0.855892,Spread of 13.8 units between best and worst gr...,Significant Disparity (Worst group travels 137...,Slightly Progressive (Most deprived travel 14%...,LSOA 2021 Name Bideford Community Hospi...,7
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 5, 8, 10, 12]",30,18.345390,16.899177,30.960000,56.350000,NaN,...,2.214925,14.7000,20.543333,16.286667,0.902579,Spread of 12.2 units between best and worst gr...,Significant Disparity (Worst group travels 121...,Slightly Progressive (Most deprived travel 10%...,LSOA 2021 Name Bideford Community Hospi...,8
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 5, 8, 10, 12, 14]",30,17.359789,16.176703,29.703333,49.550000,NaN,...,2.078607,14.6600,18.213333,16.220000,0.903822,Spread of 10.8 units between best and worst gr...,Significant Disparity (Worst group travels 108...,Slightly Progressive (Most deprived travel 10%...,LSOA 2021 Name Bideford Community Hospi...,9
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 4, 5, 8, 10, 12, 14]",30,16.402082,15.320622,28.193332,45.966667,NaN,...,1.974129,13.6250,17.190000,15.850000,0.859621,Spread of 9.8 units between best and worst groups,Significant Disparity (Worst group travels 97%...,Slightly Progressive (Most deprived travel 14%...,LSOA 2021 Name Bideford Community Hospi...,10
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 4, 5, 8, 10, 12, 14, 17]",30,15.594110,14.528441,27.423333,45.966667,NaN,...,2.196101,12.4950,16.286667,15.523333,0.804917,Spread of 10.4 units between best and worst gr...,Significant Disparity (Worst group travels 120...,Slightly Progressive (Most deprived travel 20%...,LSOA 2021 Name Bideford Community Hospi...,11
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 4, 5, 8, 10, 12, 14, 15, 17]",30,14.842127,13.930773,25.770000,45.966667,NaN,...,2.071101,12.2225,15.293333,14.696667,0.831651,Spread of 9.3 units between best and worst groups,Significant Disparity (Worst group travels 107...,Slightly Progressive (Most deprived travel 17%...,LSOA 2021 Name Bideford Community Hospi...,12
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 4, 5, 8, 10, 11, 12, 14, 15, 17]",30,14.188906,13.360517,25.553334,45.966667,NaN,...,1.987385,11.7900,14.783333,13.490000,0.873981,Spread of 8.6 units between best and worst groups,Sig

In [8]:
result_df_comparison.to_pickle("comparison_num_sites.pkl")

In [9]:
px.bar(result_df_comparison, x="n", y="weighted_average")

In [10]:
px.bar(result_df_comparison, x="n", y="max")

In [11]:
px.bar(result_df_comparison, x="n", y="90th_percentile")

## Explore solutions

In [12]:
solution_car_5 = [i for i in solutions_car if i["n"] == 5][0]['solution']

In [13]:
solution_car_5.show_solutions()

,solution_rank,site_names,site_indices,coverage_threshold,weighted_average,unweighted_average,90th_percentile,max,total_cost,proportion_within_coverage_threshold,...,gap_absolute_weighted,gap_relative_weighted,avg_lower_third_bins,avg_middle_third_bins,avg_upper_third_bins,inter_tertile_ratio,gap_absolute_description,gap_relative_description,inter_tertile_description,problem_df
0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 12]",30,21.93,20.03,39.41,61.53,NaN,0.75,...,17.66,2.76,16.97,24.68,18.69,0.91,Spread of 17.7 units between best and worst gr...,Significant Disparity (Worst group travels 176...,Slightly Progressive (Most deprived travel 9% ...,LSOA 2021 Name Bideford Community Hospi...
1,2,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 5]",30,22.05,20.37,37.61,61.53,NaN,0.75,...,14.73,2.22,17.50,24.90,19.22,0.91,Spread of 14.7 units between best and worst gr...,Significant Disparity (Worst group travels 122...,Slightly Progressive (Most deprived travel 9% ...,LSOA 2021 Name Bideford Community Hospi...
2,3,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 8]",30,22.15,20.21,42.07,61.18,NaN,0.75,...,15.81,2.31,18.12,24.62,18.04,1.00,Spread of 15.8 units between best and worst gr...,Significant Disparity (Worst group travels 131...,Balanced (Macro travel times are broadly equal),LSOA 2021 Name Bideford Community Hospi...
3,4,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 4]",30,22.19,20.39,41.07,61.53,NaN,0.75,...,15.94,2.32,17.56,24.83,18.89,0.93,Spread of 15.9 units between best and worst gr...,Significant Disparity (Worst group travels 132...,Slightly Progressive (Most deprived travel 7% ...,LSOA 2021 Name Bideford Community Hospi...
4,5,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 9]",30,22.26,20.38,41.27,61.53,NaN,0.74,...,15.37,2.27,18.30,24.58,18.61,0.98,Spread of 15.4 units between best and worst gr...,Significant Disparity (Worst group travels 127...,Balanced (Macro travel times are broadly equal),LSOA 2021 Name Bideford Community Hospi...
5,6,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 17]",30,22.26,20.37,41.91,61.18,NaN,0.73,...,17.35,2.61,17.68,24.54,18.86,0.94,Spread of 17.4 units between best and worst gr...,Significant Disparity (Worst group travels 161...,Slightly Progressive (Most deprived travel 6% ...,LSOA 2021 Name Bideford Community Hospi...
6,7,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 14]",30,22.28,20.60,39.47,61.18,NaN,0.75,...,16.66,2.38,18.77,23.50,19.20,0.98,Spread of 16.7 units between best and worst gr...,Significant Disparity (Worst group travels 138...,Balanced (Macro travel times are broadly equal),LSOA 2021 Name Bideford Community Hospi...
7,8,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 15]",30,22.28,20.54,39.94,61.53,NaN,0.74,...,15.17,2.25,18.24,24.80,18.40,0.99,Spread of 15.2 units between best and worst gr...,Significant Disparity (Worst group travels 125...,Balanced (Macro travel times are broadly equal),LSOA 2021 Name Bideford Community Hospi...
8,9,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 10]",30,22.32,20.51,39.44,61.53,NaN,0.75,...,16.23,2.34,18.66,24.45,18.12,1.03,Spread of 16.2 units between best and worst gr...,Significant Disparity (Worst group travels 134...,Balanced (Macro travel times are broadly equal),LSOA 2021 Name Bideford Community Hospi...
9,10,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 7]",30,22.42,20.69,39.84,61.53,NaN,0.74,...,16.43,2.40,17.62,25.51,19.23,0.92,Spread of 16.4 units between best and worst gr...,Significant Disparity (Worst group travels 140...,Slightly Progressive (Most deprived travel 8% ...,LSOA 2021 Name Bideford Community Hospi...


In [14]:
solution_car_5.show_solutions_colnames()

Index(['solution_rank', 'site_names', 'site_indices', 'coverage_threshold',
       'weighted_average', 'unweighted_average', '90th_percentile', 'max',
       'total_cost', 'proportion_within_coverage_threshold',
       'proportion_regions_within_coverage_threshold',
       'weighted_by_equity_group', 'unweighted_by_equity_group',
       'coverage_by_equity_group', 'coverage_regions_by_equity_group',
       'max_cost_by_equity_group', 'gap_absolute_weighted',
       'gap_relative_weighted', 'avg_lower_third_bins',
       'avg_middle_third_bins', 'avg_upper_third_bins', 'inter_tertile_ratio',
       'gap_absolute_description', 'gap_relative_description',
       'inter_tertile_description', 'problem_df'],
      dtype='str')


## Solution Outputs

In [15]:
solution_car_5 = [i for i in solutions_car if i["n"] == 5][0]['solution']

In [16]:
solution_car_5

In [17]:
with open("solution_car_5.pkl", "wb") as f:
    pickle.dump(solution_car_5, f)

In [18]:
solution_car_5.solution_df.to_pickle("solution_car_5_solution_df.pkl")

In [19]:
def generate_solution_animation(solution, output_filename, frame_duration=0.5):

    frames = []

    total_n_solutions = len(solution.solution_df)

    # Get existing sites
    sites = solution.site_problem.show_sites()
    existing_sites = sites[sites[solution.site_problem._candidate_sites_required_sites_col]=="Yes"][solution.site_problem._candidate_sites_candidate_id_col].to_list()

    for i in range(1, total_n_solutions + 1):
        # Work out which the one additional site is
        sites_in_solution = solution.solution_df[solution.solution_df["solution_rank"]==i]["site_names"].iloc[0]
        new_site = [i for i in sites_in_solution if i not in existing_sites]
        ax = solution.plot_best_combination(solution_rank=i, title=f"Trying out {new_site[0]}\n(Option {i} of {total_n_solutions})")
        fig = ax.figure

        buffer = io.BytesIO()
        fig.savefig(buffer, format="png", bbox_inches="tight")
        buffer.seek(0)

        frames.append(Image.open(buffer).copy())

        plt.close(fig)

    frames[0].save(
        f"{output_filename}.gif",
        save_all=True,
        append_images=frames[1:],
        duration=int(frame_duration * 1000),
        loop=0,
    )

In [20]:
solution_car_6 = [i for i in solutions_car if i["n"] == 6][0]['solution']

In [21]:
with open("solution_car_6.pkl", "wb") as f:
    pickle.dump(solution_car_6, f)

In [22]:
solution_car_6.solution_df.to_pickle("solution_car_6_solution_df.pkl")

## Animation Generation

In [ ]:
generate_solution_animation(solution_car_5, output_filename="solution_car_5", frame_duration=0.8)

In [ ]:
generate_solution_animation(solution_car_6, output_filename="solution_car_6", frame_duration=0.2)

In [ ]:
# solution_car_7 = [i for i in solutions_car if i["n"] == 7][0]['solution']
# generate_solution_animation(solution_car_7, output_filename="solution_car_7", frame_duration=0.1)

In [ ]:
# solution_car_8 = [i for i in solutions_car if i["n"] == 8][0]['solution']
# generate_solution_animation(solution_car_8, output_filename="solution_car_8", frame_duration=0.05)

In [ ]:
# solution_car_6.solution_df.head(10).to_pickle("solution_car_6_best.pkl")
# solution_car_7.solution_df.head(10).to_pickle("solution_car_7_best.pkl")
# solution_car_8.solution_df.head(10).to_pickle("solution_car_8_best.pkl")

In [ ]:
# solution_car_6.solution_df.tail(1).to_pickle("solution_car_6_worst.pkl")
# solution_car_7.solution_df.tail(1).to_pickle("solution_car_7_worst.pkl")
# solution_car_8.solution_df.tail(1).to_pickle("solution_car_8_worst.pkl")